In [1]:
import os


In [2]:
%pwd


'd:\\Text-summerizer\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\Text-summerizer'

In [5]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ModelTrainerConfig:

    root_dir: Path
    data_path: Path
    model_ckpt: Path
    num_train_epochs: int
    warmup_steps: int
    per_device_train_batch_size: int
    weight_decay: float
    logging_steps: int
    evaluation_strategy: str
    eval_steps: int
    save_steps: float
    gradient_accumulation_steps: int


In [6]:
from textsummarizer.constants import *
from textsummarizer.utils.common import read_yaml,create_directories

In [7]:
class configurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH, params_filepath=CONFIG_FILE_NAME):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])


    def get_model_trainer_config(self) -> ModelTrainerConfig:
                config = self.config.model_trainer
                params = self.params.TrainingArguments
    
                create_directories([config.root_dir])
    
                model_trainer_config = ModelTrainerConfig(
                    root_dir=Path(config.root_dir),
                    data_path=Path(config.data_path),
                    model_ckpt=config.model_ckpt,
                    num_train_epochs=params.num_train_epochs,
                    warmup_steps=params.warmup_steps,
                    per_device_train_batch_size=params.per_device_train_batch_size,
                    weight_decay=params.weight_decay,
                    logging_steps=params.logging_steps,
                    evaluation_strategy=params.evaluation_strategy,
                    eval_steps=params.eval_steps,
                    save_steps=params.save_steps,
                    gradient_accumulation_steps=params.gradient_accumulation_steps
                )
    
                return model_trainer_config

In [8]:
from transformers import TrainingArguments,Trainer
from transformers import DataCollatorForSeq2Seq
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from datasets import load_dataset,load_from_disk
import torch

In [9]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def train(self):

        device = "cuda" if torch.cuda.is_available() else "cpu"

        model = AutoModelForSeq2SeqLM.from_pretrained(
            self.config.model_ckpt
        ).to(device)

        tokenizer = AutoTokenizer.from_pretrained(
            self.config.model_ckpt
        )

        dataset_samsum_pt = load_from_disk(
            self.config.data_path
        )

        seq2seq_data_collator = DataCollatorForSeq2Seq(
            tokenizer=tokenizer,
            model=model
        )

        trainer_args = TrainingArguments(
            output_dir=self.config.root_dir,
            num_train_epochs=self.config.num_train_epochs,
            warmup_steps=self.config.warmup_steps,
            per_device_train_batch_size=self.config.per_device_train_batch_size,
            weight_decay=self.config.weight_decay,
            logging_steps=self.config.logging_steps,
            eval_strategy=self.config.evaluation_strategy,
            eval_steps=self.config.eval_steps,
            save_steps=self.config.save_steps,
            gradient_accumulation_steps=self.config.gradient_accumulation_steps
        )

        trainer = Trainer(
            model=model,
            args=trainer_args,
            processing_class=tokenizer,
            data_collator=seq2seq_data_collator,
            train_dataset=dataset_samsum_pt["train"],
            eval_dataset=dataset_samsum_pt["validation"]
        )

        trainer.train()

        model.save_pretrained(
            os.path.join(self.config.root_dir, "pegasus-samsum-model")
        )

        tokenizer.save_pretrained(
            os.path.join(self.config.root_dir, "tokenizer")
        )

In [11]:
try:
    config = configurationManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer_config = ModelTrainer(config = model_trainer_config)
    model_trainer_config.train()
except Exception as e:
    raise e

[2026-09-16 08:29:57,595]:INFO:textsummarizerLogger: yaml file: config\config.yaml loaded successfully
[2026-09-16 08:29:57,618]:INFO:textsummarizerLogger: yaml file: params.yaml loaded successfully
[2026-09-16 08:29:58,277]:INFO:httpx: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
[2026-09-16 08:29:58,335]:INFO:httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json?%2Fgoogle%2Fpegasus-cnn_dailymail%2Fresolve%2Fmain%2Fconfig.json=&etag=%222c1a911e577525af99c26c1634c473667e1e7ae2%22 "HTTP/1.1 200 OK"
[2026-09-16 08:29:58,699]:INFO:httpx: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
[2026-09-16 08:29:59,010]:INFO:httpx: HTTP Request: GET https://huggingface.co/api/models/google/pegasus-cnn_dailymail "HTTP/1.1 200 OK"
[2026-09

: 